# Train the DeepVoid demo models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skumagai3/deepvoid-demo/blob/main/notebooks/train_demo_models.ipynb)

Reproduces everything in `weights/` and `results/`, and the figures in `assets/`. Each training run
produces these models:


| Model | Trained on | Applied to | What it tests |
|---|---|---|---|
| `dm_model` | dark matter (λ = 0.33 h⁻¹ Mpc) | dark matter | the basic task, with the dark matter density field as input |
| `subhalo_no_retraining` | dark matter | galaxies (λ = 7 h⁻¹ Mpc) | does a dark-matter model work on sparse tracers as-is? (input rescaled, weights unchanged) |
| `subhalo_from_scratch` | galaxies | galaxies | training directly on the sparse field, starting from random weights |
| `subhalo_curriculum` | dark matter, then galaxies | galaxies | one-step curriculum learning: start from `dm_model`, freeze its first encoder block, fine-tune on galaxies |

All four are scored against the same T-web labels (computed from the full dark matter) on the same
validation subcubes.


Training setup: 32³ subcubes, an 80/20 random split of subcubes (seed 12), the four
rotations on training subcubes only, Adam, sparse categorical cross-entropy, learning-rate reduction
(×0.25 after 10 epochs without improvement) and early stopping (patience 25, best weights restored),
both on validation loss.

This notebook trains the models once for every input normalization option (and, optionally, with and
without overlapping training subcubes), shows a comparison table, and publishes the configuration you
choose. Use a GPU runtime (Runtime → Change runtime type → GPU). One configuration took 2.7 minutes
on a Colab GPU without overlap; with overlap, epochs are about 3.6× longer. The last cell downloads
`deepvoid_demo_outputs.zip`.

In [ ]:
import os, subprocess
os.environ["GIT_TERMINAL_PROMPT"] = "0"     # fail fast instead of asking for a password
if not os.path.isdir("dvdemo"):
    if os.path.isdir("../dvdemo"):
        os.chdir("..")
    elif subprocess.run(["git", "clone", "-q", "--depth", "1",
                         "https://github.com/skumagai3/deepvoid-demo.git"]).returncode == 0:
        os.chdir("deepvoid-demo")
    else:  # repo not public yet: upload the zip of the repo instead
        from google.colab import files
        print("Upload deepvoid-demo.zip")
        name = next(iter(files.upload()))
        subprocess.run(["unzip", "-q", "-o", name], check=True)
        os.chdir("deepvoid-demo")
print("working in", os.getcwd())

In [ ]:
import json, shutil, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from dvdemo.data import Normalizer, split_masks
from dvdemo.io import load_sample
from dvdemo.train import ArchConfig, StageConfig, run_all

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow", tf.__version__, "| GPUs:", gpus)
if not gpus:
    print("WARNING: no GPU - training will be very slow. Switch the runtime to GPU.")
sample = load_sample("data/tng300_128.npz")

## Choices

Two settings are open choices rather than fixed parts of the method. Both are saved in each model's
`.json`.

### 1. Input normalization

The stored fields are ρ/ρ̄ = 1 + δ. All three options are fitted on the
training voxels only and reused unchanged at prediction time.

| Option | Transform | Rationale |
|---|---|---|
| `"minmax"` | (x − min) / (max − min) | What the DeepVoid paper uses. Simple, bounded to [0, 1], no free parameters, and identical whether the input is ρ, ρ/ρ̄ or δ. |
| `"log"` | log10(1 + δ), then min-max | Cosmological density fields are close to log-normal, so the log spreads out the low-density range where voids and walls differ, and puts dense and sparse tracers on a common relative scale. Empty voxels (93% of the galaxy field) are floored at ρ/ρ̄ = 10⁻³ before the log. |
| `"log_zscore"` | log10(1 + δ), then (y − mean) / std | Same log; scaling by the mean and spread of the distribution rather than its two most extreme voxels, so a single unusually dense halo does not change the scale. |

### 2. Overlapping training subcubes

Without overlap, training uses the 51 non-overlapping training
subcubes (faster). With overlap it adds subcubes at a stride of 16 voxels
that lie entirely inside training subcubes (185 in total): more varied training data, longer epochs.
Validation subcubes are never touched either way.

The histograms show what each normalization does to the training voxels of both input fields.

In [ ]:
train_mask, _ = split_masks(sample["mask"].shape, sample["meta"]["split"])
fig, axes = plt.subplots(2, 3, figsize=(12, 5.4))
for r, (field, title) in enumerate((("dm", "Dark matter, λ = 0.33 h⁻¹ Mpc"),
                                    ("subhalo_L7", "Galaxies, λ = 7 h⁻¹ Mpc"))):
    x = sample[field][train_mask]
    for c, method in enumerate(("minmax", "log", "log_zscore")):
        v = Normalizer(method).fit(x).transform(x)
        axes[r, c].hist(v, bins=100, color="#2f5d8a", log=True)
        axes[r, c].set_title(f"{method}: {title}", fontsize=9)
        axes[r, c].spines[["top", "right"]].set_visible(False)
for a in axes[1]: a.set_xlabel("model input value")
for a in axes[:, 0]: a.set_ylabel("voxels (log scale)")
plt.tight_layout(); plt.show()

Smoke test (a few minutes): runs every stage for one epoch and writes to `/tmp`, to catch problems before the long run.

In [ ]:
tiny = dict(max_epochs=1)
_ = run_all(sample, out_dir="/tmp/smoke/weights", results_dir="/tmp/smoke/results",
            dense=StageConfig(**tiny), curriculum=StageConfig(**tiny, learning_rate=1e-4, frozen_blocks=1),
            scratch=StageConfig(**tiny))
print("smoke test passed")

## Train every configuration

Each configuration trains all three models from scratch and writes them to `runs/<configuration>/`.
Edit the two lists to run fewer.

In [ ]:
NORMS = ["minmax", "log", "log_zscore"]
OVERLAPS = [False, True]
ARCH = ArchConfig(depth=3, filters=16, batch_norm=True, dropout=0.0)

rows = []
for overlap in OVERLAPS:
    for norm in NORMS:
        tag = f"{norm}{'_overlap' if overlap else ''}"
        common = dict(max_epochs=300, batch_size=8, normalization=norm, overlap=overlap, verbose=0)
        t = time.time()
        print(f"=== {tag} ===", flush=True)
        res = run_all(sample, out_dir=f"runs/{tag}/weights", results_dir=f"runs/{tag}/results", arch=ARCH,
                      dense=StageConfig(learning_rate=1e-3, seed=0, **common),
                      curriculum=StageConfig(learning_rate=1e-4, frozen_blocks=1, seed=1, **common),
                      scratch=StageConfig(learning_rate=1e-3, seed=2, **common))
        print(f"    {(time.time() - t) / 60:.1f} min", flush=True)
        for name, r in res.items():
            m = r["val_metrics"]
            rows.append({"config": tag, "model": name, "void_f1": m["void_f1"], "mcc": m["mcc"],
                         "balanced_acc": m["balanced_accuracy"], "halo_f1": m["halo_f1"]})

summary = pd.DataFrame(rows)
summary.to_csv("runs/summary.csv", index=False)
summary.pivot(index="model", columns="config", values="mcc").round(3)

## Choose the configuration to publish

The table above shows validation MCC; the next cell shows every metric. By default the cell picks the
configuration with the highest mean validation MCC across the three trained models; set `CHOSEN`
yourself to override. Note that picking a configuration on the validation subcubes (which also drive
early stopping) makes its validation scores slightly optimistic, which the README mentions.

In [ ]:
trained = summary[summary.model != "subhalo_no_retraining"]
ranking = trained.groupby("config").mcc.mean().sort_values(ascending=False)
print("mean validation MCC of the three trained models:")
print(ranking.round(4).to_string())

CHOSEN = ranking.index[0]          # or e.g. CHOSEN = "minmax"
print("\npublishing:", CHOSEN)
for d in ("weights", "results"):
    if os.path.isdir(d):
        shutil.rmtree(d)
    shutil.copytree(f"runs/{CHOSEN}/{d}", d)
with open("results/chosen_config.json", "w") as f:
    json.dump({"chosen": CHOSEN, "rule": "highest mean validation MCC of the three trained models",
               "mean_mcc": ranking.to_dict()}, f, indent=2)
shutil.copy("runs/summary.csv", "results/all_configs.csv")
summary[summary.config == CHOSEN].set_index("model").round(3)

In [ ]:
!python scripts/make_figures.py
from IPython.display import Image, display
display(Image("assets/hero.gif"))

In [ ]:
!zip -q -r deepvoid_demo_outputs.zip weights results assets/hero.gif assets/*.png runs/*/results/metrics.json runs/summary.csv
try:
    from google.colab import files
    files.download("deepvoid_demo_outputs.zip")
except ImportError:
    print("wrote deepvoid_demo_outputs.zip")